# Notebook 06 — RUL Target Calculation

**Objective:** Compute Remaining Useful Life (RUL) for every eligible discharge cycle.

**Definitions:**
- **EOL (End-of-Life)**: first cycle where SOH ≤ 70%
- **RUL** = EOL_cycle − current_cycle  (floor at 0)

**Exclusion rule:** Batteries that NEVER reach 70% SOH are excluded from
RUL training. We do NOT fabricate an EOL cycle for them.

**Leakage note:** `EOL_cycle` is used only to compute the RUL target.
It is NOT a model feature. Current SOH IS valid as a feature for RUL.

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = os.path.dirname(os.path.abspath(''))
sys.path.insert(0, os.path.join(ROOT, 'src'))

from soh import add_soh, EOL_THRESHOLD
from rul import compute_eol_cycles, add_rul, rul_stats, EOL_SOH_THRESHOLD

sns.set_style('whitegrid')
plt.rcParams.update({'font.size': 11})

## 1. Load Featured Dataset & Add SOH

In [ ]:
feat = pd.read_csv(os.path.join(ROOT, 'cleaned_data', 'featured_dataset.csv'))
feat = feat.sort_values(['battery_id', 'cycle']).reset_index(drop=True)
feat = add_soh(feat)

print(f'Shape: {feat.shape}  |  Batteries: {feat["battery_id"].nunique()}')
print(f'SOH range: {feat["soh"].min():.2f}% – {feat["soh"].max():.2f}%')

## 2. Find EOL Cycles (SOH <= 70%)

In [ ]:
eol_cycles = compute_eol_cycles(feat)

all_bats  = sorted(feat['battery_id'].unique())
eol_bats  = sorted(eol_cycles.keys())
non_eol   = sorted(set(all_bats) - set(eol_bats))

print(f'Total batteries: {len(all_bats)}')
print(f'EOL  batteries : {len(eol_bats)}')
print(f'Non-EOL (excluded from RUL): {len(non_eol)}')
print()
print('EOL cycles per battery:')
for b, c in sorted(eol_cycles.items()):
    print(f'  {b}: EOL at cycle {c}')

## 3. Add RUL Column

In [ ]:
feat = add_rul(feat, eol_cycles)

rul_valid = feat['rul'].dropna()
print(f'Rows with valid RUL : {len(rul_valid)}  ({len(rul_valid)/len(feat)*100:.1f}%)')
print(f'Rows excluded (NaN) : {feat["rul"].isna().sum()}  (non-EOL batteries)')
print(f'RUL range: {rul_valid.min():.0f} – {rul_valid.max():.0f} cycles')
print(f'RUL mean : {rul_valid.mean():.1f} cycles')
feat[['battery_id', 'cycle', 'soh', 'rul']].dropna().head(12)

## 4. Non-EOL Battery Report

In [ ]:
print('Batteries excluded from RUL training:')
print(f'  {non_eol}')
print()
for b in non_eol:
    grp = feat[feat['battery_id'] == b]
    print(f'  {b}: {len(grp)} cycles  |  '
          f'SOH range {grp["soh"].min():.1f}% – {grp["soh"].max():.1f}%  '
          '(never reached 70% SOH threshold)')
print()
print('These batteries were tested with conservative protocols (low degradation).')
print('Inventing an EOL cycle for them would corrupt the RUL target.')

## 5. RUL Distribution

In [ ]:
rul_df = feat.dropna(subset=['rul'])

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(rul_df['rul'], bins=40, color='#C62828', edgecolor='white', linewidth=0.5)
axes[0].set_xlabel('RUL (cycles)')
axes[0].set_ylabel('Count')
axes[0].set_title('Distribution of RUL Across All EOL-Battery Cycles')

axes[1].scatter(rul_df['cycle'], rul_df['rul'],
                alpha=0.3, s=15, color='#C62828')
axes[1].set_xlabel('Discharge Cycle')
axes[1].set_ylabel('RUL (cycles)')
axes[1].set_title('RUL vs Cycle (EOL batteries only)')

plt.tight_layout()
plt.show()

## 6. RUL Timeline per Battery

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))
for bid in eol_bats:
    grp = rul_df[rul_df['battery_id'] == bid].sort_values('cycle')
    ax.plot(grp['cycle'], grp['rul'], alpha=0.7, linewidth=1.5, label=bid)

ax.axhline(0, color='black', linestyle='--', linewidth=1)
ax.set_xlabel('Discharge Cycle')
ax.set_ylabel('RUL (cycles remaining to EOL)')
ax.set_title('RUL Trajectory — All EOL Batteries')
ax.legend(fontsize=7, ncol=3, loc='upper right')
plt.tight_layout()
plt.show()

## 7. Correlation: RUL vs Features

In [ ]:
feat_cols = ['cycle', 'soh', 'discharge_duration', 'energy_Wh',
             'voltage_mean', 'voltage_min', 'voltage_std',
             'current_mean', 'temperature_mean', 'temperature_max', 'ambient_temp_C']

corrs = rul_df[feat_cols + ['rul']].corr()['rul'].drop('rul').sort_values()

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#C62828' if v < 0 else '#1565C0' for v in corrs.values]
ax.barh(corrs.index, corrs.values, color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Pearson Correlation with RUL')
ax.set_title('Feature–RUL Correlation')
plt.tight_layout()
plt.show()

## 8. Save Final Dataset

In [ ]:
FINAL_COLS = (['battery_id', 'cycle', 'capacity', 'initial_capacity', 'soh', 'rul'] +
              ['discharge_duration', 'energy_Wh', 'voltage_mean', 'voltage_min',
               'voltage_std', 'current_mean', 'temperature_mean', 'temperature_max',
               'ambient_temp_C'])

final_df = feat[FINAL_COLS].copy()
out = os.path.join(ROOT, 'cleaned_data', 'final_dataset.csv')
final_df.to_csv(out, index=False)
print(f'Saved: {out}')
print(f'Shape: {final_df.shape}')
print('Columns:', list(final_df.columns))